In [ ]:
# Install pymatgen for crystal structure and XRD calculations
!pip install pymatgen

In [ ]:
# Import pymatgen to verify that the installation works
import pymatgen

# Import the version information from pymatgen
from importlib.metadata import version

# Display the installed pymatgen version
print(version("pymatgen"))



2026.5.4


In [ ]:
# Import NumPy for numerical calculations
import numpy as np

# Import pandas for organizing our future XRD dataset
import pandas as pd

# Import NumPy for numerical calculations
import numpy as np

# Import pandas for organizing our XRD data into tables
import pandas as pd

# Import crystal-structure tools from pymatgen
from pymatgen.core import Lattice, Structure

In [ ]:
# Import pymatgen's XRD calculator for theoretical powder XRD
from pymatgen.analysis.diffraction.xrd import XRDCalculator

# Use Cu K-alpha radiation, matching a common laboratory XRD source
xrd_calculator = XRDCalculator(wavelength="CuKa")

Now we will test a single nickel FCC structure to see if the XRD calculator works


In [ ]:
# Define the FCC Ni lattice parameter in Angstrom
a = 3.52
ni_lattice = Lattice.cubic(a)

# Place Ni atoms at the four FCC positions
ni_structure = Structure(
    ni_lattice, ["Ni"] * 4,
    [[0,0,0], [0,0.5,0.5], [0.5,0,0.5], [0.5,0.5,0]]
)

# Calculate the theoretical powder XRD pattern
ni_pattern = xrd_calculator.get_pattern(ni_structure)

# Display 2θ, intensity, and d-spacing for the first three peaks
print("2θ:", ni_pattern.x[:10])
print("Intensity:", ni_pattern.y[:10])
print("d-spacing:", ni_pattern.d_hkls[:10])
# Check how many diffraction peaks were calculated
print("Number of peaks:", len(ni_pattern.x))

# Show the complete calculated 2θ range
print("2θ range:", ni_pattern.x[0], "to", ni_pattern.x[-1])
# Recalculate the Ni XRD pattern over a wider 2θ range
ni_pattern = xrd_calculator.get_pattern(
    ni_structure, two_theta_range=(10, 140)
)

# Check the number of calculated reflections
print("Number of peaks:", len(ni_pattern.x))
# Display all calculated Ni diffraction peaks
print("2θ:", ni_pattern.x)
print("Intensity:", ni_pattern.y)
print("d-spacing:", ni_pattern.d_hkls)

2θ: [44.585466   51.95558252 76.55308463]
Intensity: [100.          46.36686035  26.67333336]
d-spacing: [np.float64(2.0322729475474826), np.float64(1.7599999999999998), np.float64(1.2445079348883235)]
Number of peaks: 3
2θ range: 44.585465996421426 to 76.55308462739535
Number of peaks: 6
2θ: [ 44.585466    51.95558252  76.55308463  93.1672908   98.69551588
 122.33789463]
Intensity: [100.          46.36686035  26.67333336  32.85999733  10.00435025
   7.21936084]
d-spacing: [np.float64(2.0322729475474826), np.float64(1.7599999999999998), np.float64(1.2445079348883235), np.float64(1.061319932913728), np.float64(1.0161364737737413), np.float64(0.8799999999999999)]


## Plan Ahead

We will first create reusable functions for FCC and HCP crystal structures.
Then we will generate multiple synthetic structures for our selected
elements, calculate their XRD patterns using pymatgen, and extract the
structural features needed for classification.


In [ ]:
# Define a reusable function that creates an FCC structure
def make_fcc(element, a):

    # Create a cubic lattice using the supplied lattice parameter
    lattice = Lattice.cubic(a)

    # Place four atoms at the conventional FCC positions
    structure = Structure(
        lattice, [element] * 4,
        [[0,0,0], [0,0.5,0.5], [0.5,0,0.5], [0.5,0.5,0]]
    )

    # Return the completed FCC crystal structure
    return structure

## Materials Project Data Retrieval

We will use the Materials Project API to retrieve verified reference crystal
structures for our selected FCC and HCP elements. We will first test the
procedure using Ni before collecting data for all 12 elements.

In [ ]:
# Install the Materials Project Python API
!pip install mp-api

In [ ]:
# Import the Materials Project API client
from mp_api.client import MPRester

# Enter your Materials Project API key securely
API_KEY = input( )

In [ ]:
from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import pandas as pd

# -----------------------------
# Your 12 elemental prototypes
# -----------------------------

fcc_elements = ["Ni", "Cu", "Al", "Ag", "Pt", "Pb"]
hcp_elements = ["Co", "Zn", "Mg", "Ti", "Zr", "Cd"]

# -----------------------------
# Search Materials Project
# -----------------------------

all_data = []

with MPRester(API_KEY) as mpr:

    for phase, elements in [("FCC", fcc_elements),
                            ("HCP", hcp_elements)]:

        for prototype in elements:

            print(f"Searching {phase}: {prototype}")

            docs = mpr.materials.summary.search(
                elements=[prototype],
                num_elements=(1, 5),
                fields=[
                    "material_id",
                    "formula_pretty",
                    "structure",
                    "composition"
                ]
            )

            for doc in docs:

                try:
                    sga = SpacegroupAnalyzer(doc.structure)

                    # Convert to conventional standard cell
                    conventional = (
                        sga.get_conventional_standard_structure()
                    )

                    sg_number = sga.get_space_group_number()

                except Exception:
                    continue

                # FCC = Fm-3m (#225)
                if phase == "FCC" and sg_number != 225:
                    continue

                # HCP = P63/mmc (#194)
                if phase == "HCP" and sg_number != 194:
                    continue

                # Number of chemical elements
                nelements = len(doc.composition.elements)

                # Assign tier
                if nelements == 1:
                    tier = 1
                elif nelements <= 3:
                    tier = 2
                else:
                    tier = 3

                all_data.append([
                    phase,
                    tier,
                    prototype,
                    str(doc.material_id),
                    doc.formula_pretty,
                    nelements,
                    sg_number,
                    conventional.lattice.a,
                    conventional.lattice.b,
                    conventional.lattice.c
                ])

# -----------------------------
# Create dataframe
# -----------------------------

columns = [
    "Class",
    "Tier",
    "Prototype",
    "Materials Project ID",
    "Formula",
    "Number of Elements",
    "Space Group",
    "a (Å)",
    "b (Å)",
    "c (Å)"
]

df = pd.DataFrame(all_data, columns=columns)

print("\nTotal eligible structures:", len(df))

Searching FCC: Ni


Retrieving SummaryDoc documents:   0%|          | 0/8124 [00:00<?, ?it/s]

Searching FCC: Cu


Retrieving SummaryDoc documents:   0%|          | 0/9546 [00:00<?, ?it/s]

Searching FCC: Al


...

Searching HCP: Zr


Retrieving SummaryDoc documents:   0%|          | 0/3507 [00:00<?, ?it/s]

Searching HCP: Cd


Retrieving SummaryDoc documents:   0%|          | 0/3296 [00:00<?, ?it/s]


Total eligible structures: 3370


In [ ]:
import pandas as pd
from pymatgen.core import Composition

# 1. Define allowed metallic elements ONLY
ALLOWED_METALS = {
    "Li", "Be", "Na", "Mg", "Al", "K", "Ca", "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni",
    "Cu", "Zn", "Ga", "Rb", "Sr", "Y", "Zr", "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd",
    "In", "Sn", "Cs", "Ba", "La", "Ce", "Pr", "Nd", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho",
    "Er", "Tm", "Yb", "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Tl", "Pb", "Bi"
}

def is_pure_metallic(formula):
    """
    Returns True if every element in the chemical formula is an allowed metal.
    Excludes non-metals like O, N, F, S, Cl, P, H, C, etc.
    """
    try:
        comp = Composition(str(formula))
        elements = {el.symbol for el in comp.elements}
        return elements.issubset(ALLOWED_METALS)
    except Exception:
        return False

# 2. Filter the input DataFrame to retain ONLY pure metallic compounds
df_filtered = df[df["Formula"].apply(is_pure_metallic)].copy()

# 3. Selection Loop with Metallic Filtering
selected = []
used_mp_ids = set()

for phase, elements in {
    "FCC": fcc_elements,
    "HCP": hcp_elements
}.items():

    for prototype in elements:

        candidates = df_filtered[
            (df_filtered["Class"] == phase) &
            (df_filtered["Prototype"] == prototype)
        ].drop_duplicates("Materials Project ID")

        chosen = []

        # Always take the elemental structure (Tier 1)
        tier1 = candidates[
            (candidates["Tier"] == 1) &
            (~candidates["Materials Project ID"].isin(used_mp_ids))
        ]

        if len(tier1) == 0:
            raise ValueError(f"No valid metallic Tier-1 structure found for {phase}-{prototype}")

        chosen.append(tier1.iloc[[0]])

        # Prefer complex structures first (Tier 3 -> Tier 2)
        for tier in [3, 2]:

            remaining = 10 - sum(len(x) for x in chosen)

            if remaining <= 0:
                break

            tier_candidates = candidates[
                (candidates["Tier"] == tier) &
                (~candidates["Materials Project ID"].isin(used_mp_ids))
            ]

            take = tier_candidates.head(remaining)

            if len(take) > 0:
                chosen.append(take)

        selected_prototype = pd.concat(chosen)

        # Check whether we actually obtained 10 metallic samples
        if len(selected_prototype) < 10:
            raise ValueError(
                f"{phase}-{prototype}: only {len(selected_prototype)} valid metallic "
                f"unique structures available."
            )

        selected_prototype = selected_prototype.head(10)

        selected.extend(selected_prototype.to_dict("records"))

        used_mp_ids.update(
            selected_prototype["Materials Project ID"]
        )

final_selected_df = pd.DataFrame(selected)
print("Total valid metallic structures selected:", len(final_selected_df))

Total valid metallic structures selected: 120


In [ ]:
final_df = pd.DataFrame(selected)

final_df = final_df[
    [
        "Class",
        "Tier",
        "Prototype",
        "Formula",
        "Materials Project ID"
    ]
]

print("Total samples:", len(final_df))
display(final_df)

Total samples: 120


,Class,Tier,Prototype,Formula,Materials Project ID
0,FCC,1,Ni,Ni,mp-23
1,FCC,2,Ni,LiAl2Ni,mp-862318
2,FCC,2,Ni,Sc2NiIr,mp-862360
3,FCC,2,Ni,Sc2NiOs,mp-862361
4,FCC,2,Ni,Sc2NiRu,mp-862367
...,...,...,...,...,...
115,HCP,2,Cd,Sm3Cd,mp-979339
116,HCP,2,Cd,Na3Cd,mp-983509
117,HCP,2,Cd,V3Cd,mp-983607
118,HCP,2,Cd,CeTlCd,mp-1018668


In [ ]:
output_file = "MP_120_FCC_HCP_Structures.xlsx"

final_df.to_excel(output_file, index=False)

from google.colab import files
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Now we will derieve the XRD data from the 120 samples:

In [ ]:
import pandas as pd
import numpy as np
from mp_api.client import MPRester
from pymatgen.analysis.diffraction.xrd import XRDCalculator

input_excel = "MP_120_FCC_HCP_Structures.xlsx"
df = pd.read_excel(input_excel)

xrd_calc = XRDCalculator(wavelength="CuKa")
API_KEY = "uqAvEMC4Pw9buze8i08DVTzORiVd5Omv"

features_list = []

print(f"Starting XRD feature extraction for {len(df)} structures...\n")

with MPRester(API_KEY) as mpr:
    for idx, row in df.iterrows():
        mp_id = str(row["Materials Project ID"]).strip()
        phase = row["Class"]
        formula = row["Formula"]
        prototype = row["Prototype"]
        tier = row["Tier"]

        try:
            structure = mpr.get_structure_by_material_id(mp_id)
            pattern = xrd_calc.get_pattern(structure)

            # 1. Get top 2 most intense peak indices
            top2_indices = np.argsort(pattern.y)[::-1][:2]

            p1_idx = top2_indices[0]
            p2_idx = top2_indices[1]

            d_p1, I_p1 = pattern.d_hkls[p1_idx], pattern.y[p1_idx]
            d_p2, I_p2 = pattern.d_hkls[p2_idx], pattern.y[p2_idx]

            # 2. ENFORCE GEOMETRIC ORDERING: d1 > d2 (d1 is always the larger d-spacing)
            if d_p1 >= d_p2:
                d1, I1 = d_p1, I_p1
                d2, I2 = d_p2, I_p2
            else:
                d1, I1 = d_p2, I_p2
                d2, I2 = d_p1, I_p1

            # 3. Calculate features
            d_ratio_sq = (d1 / d2) ** 2
            intensity_ratio = I2 / I1 if I1 != 0 else 0.0

            features_list.append({
                "Materials Project ID": mp_id,
                "Formula": formula,
                "Prototype": prototype,
                "Tier": tier,
                "Phase": phase,
                "d1 (Å)": round(d1, 4),
                "d2 (Å)": round(d2, 4),
                "I1": round(I1, 2),
                "I2": round(I2, 2),
                "(d1/d2)^2": round(d_ratio_sq, 4),
                "I2/I1": round(intensity_ratio, 4)
            })

            print(f"[{idx+1}/{len(df)}] {mp_id} ({formula}) -> Success | (d1/d2)^2 = {d_ratio_sq:.4f}")

        except Exception as e:
            print(f"[{idx+1}/{len(df)}] Error processing {mp_id}: {e}")

# Save full and minimal datasets
features_df = pd.DataFrame(features_list)
features_df.to_excel("extracted_xrd_features_full.xlsx", index=False)

ml_ready_df = features_df[["Phase", "(d1/d2)^2", "I2/I1"]]
ml_ready_df.to_csv("hea_phase_ml_dataset.csv", index=False)

print("\nOption A Feature Extraction Complete!")
print(ml_ready_df.head(10))

Starting XRD feature extraction for 120 structures...



Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

[1/120] mp-23 (Ni) -> Success | (d1/d2)^2 = 1.3333


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

[2/120] mp-862318 (LiAl2Ni) -> Success | (d1/d2)^2 = 2.6667


...

[118/120] mp-983607 (V3Cd) -> Success | (d1/d2)^2 = 2.3684


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

[119/120] mp-1018668 (CeTlCd) -> Success | (d1/d2)^2 = 1.1654


Retrieving MaterialsDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

[120/120] mp-1018753 (LaTlCd) -> Success | (d1/d2)^2 = 1.3565

Option A Feature Extraction Complete!
  Phase  (d1/d2)^2   I2/I1
0   FCC     1.3333  0.6164
1   FCC     2.6667  1.5757
2   FCC     1.0000  1.0000
3   FCC     1.0000  1.0000
4   FCC     1.0001  0.9999
5   FCC     3.0000  0.3279
6   FCC     3.0000  0.3067
7   FCC     1.0000  1.0000
8   FCC     3.0000  0.3520
9   FCC     3.0000  0.3071


Synthetic HEA dataset production:


In [ ]:
!pip install mp-api


In [ ]:
from mp_api.client import MPRester

API_KEY = "uqAvEMC4Pw9buze8i08DVTzORiVd5Omv"

with MPRester(API_KEY) as mpr:

    fcc_results = mpr.materials.summary.search(
        spacegroup_number=225,
        num_elements=(4, 6),
        is_metal=True,
        fields=[
            "material_id",
            "formula_pretty",
            "symmetry",
            "structure",
            "composition"
        ]
    )

    hcp_results = mpr.materials.summary.search(
        spacegroup_number=194,
        num_elements=(4, 6),
        is_metal=True,
        fields=[
            "material_id",
            "formula_pretty",
            "symmetry",
            "structure",
            "composition"
        ]
    )

print("\nFCC candidates:", len(fcc_results))
print("HCP candidates:", len(hcp_results))

print("\n--- FCC ---")
for doc in fcc_results[:10]:
    print(doc.material_id, "|", doc.formula_pretty)

print("\n--- HCP ---")
for doc in hcp_results[:10]:
    print(doc.material_id, "|", doc.formula_pretty)

Retrieving SummaryDoc documents:   0%|          | 0/938 [00:00<?, ?it/s]

Retrieving SummaryDoc documents:   0%|          | 0/73 [00:00<?, ?it/s]


FCC candidates: 938
HCP candidates: 73

--- FCC ---
mp-6645 | Ba14Na14CaN6
mp-645662 | Ba14Na14LiN6
mp-1095109 | Ba2BiIrO6
mp-1520718 | Ba2BiWO6
mp-1214698 | Ba2CaBiO6
mp-20841 | Ba2CaIrO6
mp-1214679 | Ba2CaNbO6
mp-6739 | Ba2CaOsO6
mp-6635 | Ba2CaReO6
mp-1206134 | Ba2CdOsO6

--- HCP ---
mp-1201626 | Al3Co(W3C)3
mp-561147 | Ba2BiRuO6
mp-556391 | Ba2Mn2Bi2O
mp-19213 | Ba2Mn2Sb2O
mp-6443 | Ba3CaRu2O9
mp-6301 | Ba3CeRu2O9
mp-1214637 | Ba3CoIr2O9
mp-19058 | Ba3CoRu2O9
mp-19337 | Ba3CoSb2O9
mp-557755 | Ba3Cr2MoO9


In [ ]:
from mp_api.client import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import pandas as pd

print("Searching broad 4–6 element FCC/HCP Materials Project structures...")

with MPRester(API_KEY) as mpr:

    fcc_docs = mpr.materials.summary.search(
        spacegroup_number=225,
        num_elements=(4, 6),
        fields=[
            "material_id",
            "formula_pretty",
            "composition",
            "structure",
            "energy_above_hull"
        ]
    )

    hcp_docs = mpr.materials.summary.search(
        spacegroup_number=194,
        num_elements=(4, 6),
        fields=[
            "material_id",
            "formula_pretty",
            "composition",
            "structure",
            "energy_above_hull"
        ]
    )

print("FCC candidates:", len(fcc_docs))
print("HCP candidates:", len(hcp_docs))
